# 04_text_features — Text Embeddings (Colab)

Turn the listing text (opportunities / risks / summary) into numeric vectors,
then reduce dimensionality with PCA. Only the **embeddings** (numbers) are saved —
the raw text stays private.

**Run this in Google Colab** (uses the free GPU for faster embedding).

**Before running**
1. Upload the private text CSV to your Drive: `MyDrive/workshop-2026/text_private_YYYY-MM-DD.csv`
2. Runtime -> Change runtime type -> Hardware accelerator: **GPU** (T4 is fine)

**Output**
- `MyDrive/workshop-2026/text_embeddings_YYYY-MM-DD.csv` (id + reduced embedding columns)
  Download this back to your local `data/processed/` for step 06.


## 1. Install & Mount

Install sentence-transformers, then mount Google Drive to read the text file.


In [ ]:
# Install the embedding library (Colab doesn't have it by default)
!pip install -q sentence-transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/workshop-2026')
print('Drive folder exists?:', DRIVE_DIR.exists())
print('Files:')
for p in sorted(DRIVE_DIR.glob('*.csv')):
    print('  -', p.name)


## 2. Load & Prepare Text

Read the private text CSV. The `opportunities` / `risks` fields are stored as
stringified lists, so we join them into sentences, then combine with `summary`
into one text blob per listing.


In [ ]:
import ast
import pandas as pd

# Pick the most recent text file
text_files = sorted(DRIVE_DIR.glob('text_private_*.csv'))
assert text_files, 'No text_private_*.csv found in Drive folder. Upload it first.'
df = pd.read_csv(text_files[-1])
print('Loaded:', text_files[-1].name, '->', df.shape)

def join_list(x):
    """Turn a stringified list into a space-joined sentence."""
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list):
            return ' '.join(str(i) for i in lst)
    except Exception:
        pass
    return str(x) if pd.notna(x) else ''

for col in ['opportunities', 'risks']:
    if col in df.columns:
        df[f'{col}_text'] = df[col].apply(join_list)

# Combine summary + opportunities + risks into one text per listing
combined = df['summary'].fillna('').astype(str)
for col in ['opportunities_text', 'risks_text']:
    if col in df.columns:
        combined = combined + ' ' + df[col].fillna('').astype(str)
df['combined_text'] = combined.str.strip()

print('Combined text length (chars):')
print(df['combined_text'].str.len().describe().round(0))


## 3. Generate Embeddings

Use a compact, strong sentence-transformer model (`all-MiniLM-L6-v2`, 384 dims).
On GPU this encodes a few thousand listings in well under a minute.


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

embeddings = model.encode(
    df['combined_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print('Embeddings shape:', embeddings.shape)   # (n_listings, 384)


## 4. Reduce Dimensionality (PCA)

384 dims is a lot for ~2,800 rows -> overfitting risk when fused with tabular
features. Reduce to a smaller set of components that keep most of the variance.
(Doc guidance: embed -> PCA -> fuse with tabular in a later step.)


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Keep enough components to explain ~90% of variance, capped at 50
pca_full = PCA(n_components=min(50, embeddings.shape[0], embeddings.shape[1]))
reduced_full = pca_full.fit_transform(embeddings)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_keep = int(np.searchsorted(cumvar, 0.90) + 1)
n_keep = max(5, min(n_keep, reduced_full.shape[1]))

reduced = reduced_full[:, :n_keep]
print(f'Kept {n_keep} components explaining {cumvar[n_keep-1]*100:.1f}% of variance')
print('Reduced shape:', reduced.shape)


## 5. Save Embeddings (numbers only)

Save `id` + reduced embedding columns. No raw text is written, so this file
is safe to move around and use as a feature source in step 06.


In [ ]:
import datetime

emb_df = pd.DataFrame(
    reduced,
    columns=[f'text_emb_{i}' for i in range(reduced.shape[1])],
)
emb_df.insert(0, 'id', df['id'].values)

today = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d')
out_path = DRIVE_DIR / f'text_embeddings_{today}.csv'
emb_df.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Shape:', emb_df.shape)
emb_df.head()


## 6. Next

- Download `text_embeddings_YYYY-MM-DD.csv` from Drive to your local
  `data/processed/` folder.
- **Next: `06_model_text_fusion`** (local) — join these embeddings with the
  tabular features by `id`, retrain, and check whether text improves over the
  tabular-only baseline (the project's central claim).
